# 004. Build lexical analysis data

Canonical lexical-analysis stage for the paper pipeline.


## Purpose, inputs, and outputs

**Purpose**
- Build the cleaned abstract corpus used for lexical comparison.
- Produce inspection-level TF-IDF and keyness tables.
- Write the canonical yearly unigram table used by the manuscript figure pipeline.

**Reads**
- `data/papers.parquet`
- `data/paper_arxiv_classes.parquet`

**Writes**
- `data/unigram_yearly.parquet` (canonical manuscript input)
- support CSV/parquet outputs under `code/stage-outputs/004-build-lexical-data/support/`
- cleaned-corpus cache under `code/stage-outputs/004-build-lexical-data/cache/`


In [ ]:
from __future__ import annotations

import re
import json
import hashlib
import importlib
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

tqdm.pandas()


In [ ]:
from shared.project_paths import get_project_paths

paths = get_project_paths()
CODE_ROOT = paths.code_root
PROJECT_ROOT = paths.project_root
DATA_DIR = paths.data_dir
OUTPUT_ROOT = paths.output_dir("004-build-lexical-data")
CACHE_DIR = OUTPUT_ROOT / "cache"
SUPPORT_DIR = OUTPUT_ROOT / "support"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SUPPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from tfidf import preproc_utils
import shared.normalization as normalization
importlib.reload(normalization)
importlib.reload(preproc_utils)

from shared.normalization import TARGET, iter_sentences_norm

preproc = preproc_utils.preproc

PAPERS_PATH = DATA_DIR / "papers.parquet"
ARXIV_CLASSES_PATH = DATA_DIR / "paper_arxiv_classes.parquet"
UNIGRAM_YEARLY_PARQUET = DATA_DIR / "unigram_yearly.parquet"

MASTER_PARQUET = CACHE_DIR / "master_abstract_corpus.parquet"
MASTER_MANIFEST = CACHE_DIR / "master_abstract_corpus_manifest.json"
FIELD_TFIDF_UNIGRAM_PATH = SUPPORT_DIR / "nature_field_unigram_terms.csv"
FIELD_TFIDF_BIGRAM_PATH = SUPPORT_DIR / "nature_field_bigram_terms.csv"
KEYNESS_UNIGRAMS_PATH = SUPPORT_DIR / "keyness_unigrams.csv"
KEYNESS_BIGRAMS_PATH = SUPPORT_DIR / "keyness_bigrams.csv"
BIGRAM_YEARLY_CSV = SUPPORT_DIR / "ngram_yearly.csv"
BIGRAM_YEARLY_PARQUET = SUPPORT_DIR / "ngram_yearly.parquet"
UNIGRAM_YEARLY_CSV = SUPPORT_DIR / "unigram_yearly.csv"

TFIDF_MODEL_DIR = CODE_ROOT / "tfidf"
TFIDF_CONFIG_PATH = TFIDF_MODEL_DIR / "tfidf_config.json"
TFIDF_STOPWORDS_PATH = TFIDF_MODEL_DIR / "final_stopwords.txt"

FORCE_REBUILD_MASTER = False
END_YEAR = 2025
ABS_MIN_WORDS = 12
ABS_MAX_WORDS = 1200
TFIDF_MIN_DF = 8
TFIDF_MAX_DF = 0.35
TOP_TERMS_PER_GROUP = 25
NATURE_YEAR_MIN = 1992
FILTER_MATH_FOR_TFIDF = True
FIELD_LEVELS = ["astrophysics", "high-energy physics"]

print("PAPERS_PATH:", PAPERS_PATH)
print("ARXIV_CLASSES_PATH:", ARXIV_CLASSES_PATH)
print("MASTER_PARQUET:", MASTER_PARQUET)
print("MASTER_MANIFEST:", MASTER_MANIFEST)
print("CANONICAL_UNIGRAM_PARQUET:", UNIGRAM_YEARLY_PARQUET)
print("SUPPORT_DIR:", SUPPORT_DIR)


## Stage 1. Load or rebuild the cleaned abstract corpus


In [ ]:
if MASTER_PARQUET.exists() and MASTER_MANIFEST.exists() and not FORCE_REBUILD_MASTER:
    master_df = pd.read_parquet(MASTER_PARQUET)
    with open(MASTER_MANIFEST, 'r', encoding='utf-8') as f:
        master_manifest = json.load(f)
    print(f'Loaded cached abstract corpus -> {MASTER_PARQUET}')
    print(f'Loaded cached manifest -> {MASTER_MANIFEST}')
else:
    papers_df = pd.read_parquet(PAPERS_PATH)
    lab = pd.read_parquet(ARXIV_CLASSES_PATH)

    primary_lab = (
        lab.assign(arxiv_category=lab['arxiv_category'].astype('string'))
        .sort_values(['bibcode', 'class_pos'], kind='mergesort')
        .groupby('bibcode', as_index=False)
        .agg(
            arxiv_category=('arxiv_category', 'first'),
            primary_arxiv_class=('arxiv_class', 'first'),
            n_arxiv_classes=('arxiv_class', 'nunique'),
        )
    )

    df = papers_df.merge(primary_lab, on='bibcode', how='left')
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    mask = df['year'].notna() & df['abstract'].notna()
    df = df.loc[mask].assign(abstract=df.loc[mask, 'abstract'].astype('string').str.lower()).copy()
    df['year'] = df['year'].astype(int)

    bad_arx = [
        'quantitative finance',
        'quantitative biology',
        'economics',
        'electrical engineering and systems science',
    ]
    bad_doc = [
        'pressrelease', 'mastersthesis', 'misc', 'techreport', 'dataset', 'proposal', 'proceedings',
        'abstract', 'newsletter', 'obituary', 'talk', 'circular', 'bookreview', 'software',
        'editorial', 'erratum', 'phdthesis',
    ]

    df = df[~df['arxiv_category'].isin(bad_arx)].copy()
    df = df[~df['doctype'].isin(bad_doc)].copy()

    biology_terms_strict = {
        'microbial', 'bacterial', 'genome', 'genomes', 'eukaryotic', 'genomic',
        'microbes', 'microorganisms', 'bacteria', 'metabolites', 'phyla', 'mags', 'metagenomics'
    }
    biology_pattern = re.compile(r"\b(?:%s)\b" % '|'.join(map(re.escape, biology_terms_strict)), flags=re.I)
    df = df.loc[~df['abstract'].astype('string').str.contains(biology_pattern, na=False)].copy()

    df = df.loc[df['year'] <= END_YEAR].copy()

    df = (
        df.assign(
            bibcode=df['bibcode'].astype('string'),
            abstract=df['abstract'].astype('string'),
            arxiv_category=df['arxiv_category'].astype('string'),
            doctype=df['doctype'].astype('string'),
            primary_arxiv_class=df['primary_arxiv_class'].astype('string'),
        )
        .reset_index(drop=False)
        .rename(columns={'index': 'source_row_index'})
    )

    print('Filtered source rows:', len(df))
    df.head(3)


## Stage 2. Abstract-cleaning helpers


In [ ]:
def build_clean_abstract_record(abstract: str, bibcode: str = '') -> dict | None:
    sents = list(iter_sentences_norm(abstract))
    if not sents:
        return None

    text = ' '.join(sents).strip()
    if not text:
        return None

    n_words = len(text.split())
    n_math_tokens = len(re.findall(r'\bMATH_TOKEN\b', text))

    if n_words < ABS_MIN_WORDS:
        return None
    if n_words > ABS_MAX_WORDS:
        return None

    return {
        'abstract_clean': text,
        'bibcode': bibcode,
        'has_math': bool(n_math_tokens > 0),
        'n_words': int(n_words),
        'n_sentences': int(len(sents)),
        'n_math_tokens': int(n_math_tokens),
    }


def abstract_key(source_row_index: int, bibcode: str) -> str:
    return f'{bibcode}|||{int(source_row_index)}'


def hash_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


## Stage 3. Materialize the cleaned corpus and inspect it


In [ ]:
if 'master_df' not in globals():
    keep_cols = ['source_row_index', 'year', 'bibcode', 'abstract', 'arxiv_category', 'doctype']
    df_sub = df.loc[:, keep_cols].copy()

    master_rows = []
    for row in tqdm(df_sub.itertuples(index=False), total=len(df_sub), desc='Build cleaned abstracts'):
        rec = build_clean_abstract_record(row.abstract, bibcode=str(row.bibcode))
        if rec is None:
            continue

        rec['source_row_index'] = int(row.source_row_index)
        rec['year'] = int(row.year)
        rec['abstract_key'] = abstract_key(int(row.source_row_index), str(row.bibcode))
        rec['arxiv_category'] = None if pd.isna(row.arxiv_category) else str(row.arxiv_category)
        rec['doctype'] = None if pd.isna(row.doctype) else str(row.doctype)
        master_rows.append(rec)

    master_df = pd.DataFrame(master_rows)

    if master_df.empty:
        raise ValueError('Abstract corpus build produced no rows.')

    master_df['abstract_clean'] = master_df['abstract_clean'].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()

    master_df = (
        master_df
        .sort_values(['source_row_index', 'bibcode'], kind='mergesort')
        .drop_duplicates(subset=['abstract_key'], keep='first')
        .reset_index(drop=True)
    )
    master_df['abstract_id'] = np.arange(len(master_df), dtype=int)

    master_df = master_df[[
        'abstract_id', 'abstract_key', 'source_row_index', 'year',
        'bibcode', 'arxiv_category', 'doctype', 'abstract_clean',
        'n_words', 'n_sentences', 'n_math_tokens', 'has_math'
    ]].copy()

    master_df.to_parquet(MASTER_PARQUET, index=False)

    master_manifest = {
        'master_run_id': MASTER_RUN_ID,
        'source_papers_path': str(PAPERS_PATH),
        'source_papers_sha256': hash_file(PAPERS_PATH),
        'source_arxiv_classes_path': str(ARXIV_CLASSES_PATH),
        'source_arxiv_classes_sha256': hash_file(ARXIV_CLASSES_PATH),
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'output_parquet': str(MASTER_PARQUET),
        'force_rebuild_master': FORCE_REBUILD_MASTER,
        'filters': {
            'require_year': True,
            'require_abstract': True,
            'lowercase_abstract': True,
            'excluded_arxiv_categories': bad_arx,
            'excluded_doctypes': bad_doc,
            'biology_filter_terms': sorted(biology_terms_strict),
            'end_year': END_YEAR,
        },
        'normalization_settings': {
            'target': TARGET,
            'normalizer': 'shared.normalization.iter_sentences_norm',
            'join_strategy': 'normalized sentences joined with single spaces',
            'abs_min_words': ABS_MIN_WORDS,
            'abs_max_words': ABS_MAX_WORDS,
            'math_masking': True,
        },
        'row_counts': {
            'filtered_source_rows': int(len(df)),
            'cleaned_abstract_rows': int(len(master_df)),
        },
        'master_parquet_sha256': hash_file(MASTER_PARQUET),
    }

    with open(MASTER_MANIFEST, 'w', encoding='utf-8') as f:
        json.dump(master_manifest, f, indent=2)

    print(f'Saved abstract corpus -> {MASTER_PARQUET}')
    print(f'Saved abstract manifest -> {MASTER_MANIFEST}')

master_df.head(5)


In [ ]:
summary = {
    'n_abstracts': int(len(master_df)),
    'n_unique_bibcodes': int(master_df['bibcode'].nunique()),
    'year_min': int(master_df['year'].min()),
    'year_max': int(master_df['year'].max()),
    'has_math_rate': float(master_df['has_math'].mean()),
    'mean_words': float(master_df['n_words'].mean()),
    'median_words': float(master_df['n_words'].median()),
}
display(pd.Series(summary, name='value').to_frame())

year_counts = master_df.groupby('year').size().rename('n_abstracts').reset_index()
display(year_counts.head(20))

word_quantiles = master_df['n_words'].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).rename('n_words')
display(word_quantiles.to_frame())

category_counts = (
    master_df['arxiv_category']
    .fillna('missing')
    .value_counts(dropna=False)
    .rename_axis('arxiv_category')
    .reset_index(name='n_abstracts')
)
print(category_counts.head(15))


## Stage 4. TF-IDF comparison tables


In [ ]:
def load_stopwords(path: Path) -> set[str]:
    return {line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()}

STOPWORDS = load_stopwords(TFIDF_STOPWORDS_PATH)
TFIDF_CONFIG = json.loads(TFIDF_CONFIG_PATH.read_text(encoding='utf-8'))
TOK_PAT = TFIDF_CONFIG['TOK_PAT']

def build_field_term_table(docs: pd.Series, group_labels: pd.Series, mode: str, min_df: int, max_df: float, top_n: int) -> pd.DataFrame:
    if mode not in {'unigrams', 'bigrams'}:
        raise ValueError(f'Unsupported mode: {mode}')
    ngram_range = (1, 1) if mode == 'unigrams' else (2, 2)
    vec = TfidfVectorizer(
        preprocessor=preproc,
        token_pattern=TOK_PAT,
        stop_words=list(STOPWORDS),
        min_df=min_df,
        max_df=max_df,
        ngram_range=ngram_range,
    )
    X = vec.fit_transform(docs.fillna(''))
    terms = vec.get_feature_names_out()
    labels_arr = group_labels.to_numpy()
    mu_all = np.asarray(X.mean(axis=0)).ravel()
    N = X.shape[0]
    rows = []
    for group in sorted(pd.unique(group_labels)):
        mask = labels_arr == group
        n_in = int(mask.sum())
        if n_in == 0:
            continue
        mu_in = np.asarray(X[mask].mean(axis=0)).ravel()
        mu_out = (N * mu_all - n_in * mu_in) / max(1, (N - n_in))
        salience = mu_in - mu_out
        order = np.argsort(salience)[::-1]
        rank = 0
        for idx in order:
            if salience[idx] <= 0:
                break
            rank += 1
            rows.append({
                'group': str(group),
                'mode': mode,
                'term': str(terms[idx]),
                'salience': float(salience[idx]),
                'mean_in': float(mu_in[idx]),
                'mean_out': float(mu_out[idx]),
                'rank': int(rank),
            })
            if rank >= top_n:
                break
    return pd.DataFrame(rows)

print('TFIDF_CONFIG_PATH:', TFIDF_CONFIG_PATH)
print('TFIDF_STOPWORDS_PATH:', TFIDF_STOPWORDS_PATH)
print('Loaded stopwords:', len(STOPWORDS))
print('Token pattern:', TOK_PAT)


In [ ]:
nature_docs_df = master_df[master_df['arxiv_category'].isin(FIELD_LEVELS)].copy()
nature_docs_df = nature_docs_df[nature_docs_df['year'].between(NATURE_YEAR_MIN, END_YEAR, inclusive='both')].copy()
if FILTER_MATH_FOR_TFIDF:
    nature_docs_df = nature_docs_df[~nature_docs_df['has_math']].copy()

nature_docs_df = nature_docs_df.reset_index(drop=True)
nature_docs_df['field_doc_id'] = np.arange(len(nature_docs_df), dtype=int)

print('Nature comparison docs:', len(nature_docs_df))
display(
    nature_docs_df['arxiv_category']
    .value_counts(dropna=False)
    .rename_axis('arxiv_category')
    .reset_index(name='n_docs')
)

display(nature_docs_df[['year', 'arxiv_category', 'n_words', 'n_sentences']].head())


In [ ]:
field_unigram_terms_df = build_field_term_table(
    nature_docs_df['abstract_clean'],
    nature_docs_df['arxiv_category'],
    'unigrams',
    TFIDF_MIN_DF,
    TFIDF_MAX_DF,
    TOP_TERMS_PER_GROUP,
)
field_bigram_terms_df = build_field_term_table(
    nature_docs_df['abstract_clean'],
    nature_docs_df['arxiv_category'],
    'bigrams',
    TFIDF_MIN_DF,
    TFIDF_MAX_DF,
    TOP_TERMS_PER_GROUP,
)

field_unigram_terms_df.to_csv(FIELD_TFIDF_UNIGRAM_PATH, index=False)
field_bigram_terms_df.to_csv(FIELD_TFIDF_BIGRAM_PATH, index=False)

print(f'Saved unigram terms -> {FIELD_TFIDF_UNIGRAM_PATH}')
print(f'Saved bigram terms -> {FIELD_TFIDF_BIGRAM_PATH}')

for field in FIELD_LEVELS:
    print(f'\nField: {field}')
    print('Unigrams:')
    display(
        field_unigram_terms_df[field_unigram_terms_df['group'] == field]
        .sort_values('rank')
        .head(15)
    )
    print('Bigrams:')
    display(
        field_bigram_terms_df[field_bigram_terms_df['group'] == field]
        .sort_values('rank')
        .head(15)
    )


## Stage 5. Keyness and yearly term-rate tables


In [ ]:
if 'lab' not in globals():
    lab = pd.read_parquet(ARXIV_CLASSES_PATH)

primary_field = (
    lab.assign(arxiv_category=lab['arxiv_category'].astype('string').str.strip().str.lower())
    .sort_values(['bibcode', 'class_pos'], kind='mergesort')
    .groupby('bibcode', as_index=False)
    .agg(primary_field=('arxiv_category', 'first'), n_arxiv_classes=('arxiv_class', 'nunique'))
 )
primary_field = primary_field[primary_field['primary_field'].isin(FIELD_LEVELS)].copy()

keyness_docs_df = (
    master_df[['bibcode', 'year', 'abstract_clean', 'has_math', 'n_words', 'n_sentences']]
    .rename(columns={'abstract_clean': 'doc_text'})
    .merge(primary_field, on='bibcode', how='left')
 )
keyness_docs_df = keyness_docs_df[(keyness_docs_df['year'] >= 1992) & (keyness_docs_df['year'] <= 2025)].copy()
if FILTER_MATH_FOR_TFIDF:
    keyness_docs_df = keyness_docs_df[~keyness_docs_df['has_math']].copy()
keyness_docs_df = keyness_docs_df[keyness_docs_df['primary_field'].isin(FIELD_LEVELS) & (keyness_docs_df['n_words'] >= 8)].reset_index(drop=True)
keyness_docs_df['field_label'] = keyness_docs_df['primary_field'].replace({'astrophysics':'Astrophysics','high-energy physics':'High-energy physics'})
keyness_docs_df['doc_id'] = np.arange(len(keyness_docs_df), dtype=int)

def compute_keyness_table(df_docs: pd.DataFrame, mode: str, min_df: int):
    if mode not in {'unigrams', 'bigrams'}:
        raise ValueError(mode)
    ngram_range = (1, 1) if mode == 'unigrams' else (2, 2)
    vec = CountVectorizer(
        preprocessor=preproc,
        token_pattern=TOK_PAT,
        stop_words=list(STOPWORDS),
        min_df=min_df,
        max_df=TFIDF_MAX_DF,
        ngram_range=ngram_range,
        binary=True,
    )
    X = vec.fit_transform(df_docs['doc_text'].fillna(''))
    terms = vec.get_feature_names_out()
    label_series = df_docs['field_label'].reset_index(drop=True)
    counts = pd.DataFrame(X.toarray(), columns=terms)
    counts['field_label'] = label_series.to_numpy()
    df_counts = counts.groupby('field_label').sum().T.reset_index().rename(columns={'index':'term'})
    n_docs = label_series.value_counts()
    out = df_counts.copy()
    out['total_df'] = out['Astrophysics'] + out['High-energy physics']
    out = out[out['total_df'] >= min_df].copy()
    out['p_astro'] = (out['Astrophysics'] + 0.5) / (n_docs['Astrophysics'] + 1)
    out['p_hep'] = (out['High-energy physics'] + 0.5) / (n_docs['High-energy physics'] + 1)
    out['log_odds_astro'] = np.log2(out['p_astro'] / out['p_hep'])
    out['log_odds_hep'] = -out['log_odds_astro']
    out['salience'] = np.abs(out['log_odds_astro']) * np.log1p(out['total_df'])
    return out.sort_values('salience', ascending=False).reset_index(drop=True)

key_uni = compute_keyness_table(keyness_docs_df, 'unigrams', min_df=80)
key_bi = compute_keyness_table(keyness_docs_df, 'bigrams', min_df=40)
key_uni.to_csv(KEYNESS_UNIGRAMS_PATH, index=False)
key_bi.to_csv(KEYNESS_BIGRAMS_PATH, index=False)
display(key_bi.head(20))


In [ ]:
# document-level binary occurrence per term, per year, per field

vec = CountVectorizer(
    preprocessor=preproc,
    token_pattern=TOK_PAT,
    stop_words=list(STOPWORDS),
    min_df=40, max_df=TFIDF_MAX_DF,
    ngram_range=(2,2), binary=True,
)
X = vec.fit_transform(keyness_docs_df['doc_text'].fillna(''))
terms = vec.get_feature_names_out()

# papers per (year, field)
n_per = (keyness_docs_df.groupby(['year','field_label']).size()
         .rename('n_papers').reset_index()
         .rename(columns={'field_label': 'field'}))

# papers using term per (year, field)
df_long = (pd.DataFrame.sparse.from_spmatrix(X, columns=terms)
           .assign(year=keyness_docs_df['year'].values,
                   field=keyness_docs_df['field_label'].values)
           .groupby(['year','field'])
           .sum()
           .reset_index()
           .melt(id_vars=['year','field'], var_name='term', value_name='n_kw'))

# Arrow parquet does not support pandas Sparse columns; materialize before export.
df_long['n_kw'] = df_long['n_kw'].astype('int64')

# rate
df_long = df_long.merge(n_per, on=['year','field']).assign(rate=lambda d: d['n_kw']/d['n_papers'])
df_long.to_csv(BIGRAM_YEARLY_CSV, index=False)
df_long.to_parquet(BIGRAM_YEARLY_PARQUET, index=False)
print('Saved bigram yearly tables ->', BIGRAM_YEARLY_CSV, 'and', BIGRAM_YEARLY_PARQUET)


In [ ]:
vec = CountVectorizer(
    preprocessor=preproc,
    token_pattern=TOK_PAT,
    stop_words=list(STOPWORDS),
    min_df=80, max_df=TFIDF_MAX_DF,
    ngram_range=(1,1), binary=True,
)
X = vec.fit_transform(keyness_docs_df['doc_text'].fillna(''))
terms = vec.get_feature_names_out()

# papers per (year, field)
n_per = (keyness_docs_df.groupby(['year','field_label']).size()
         .rename('n_papers').reset_index()
         .rename(columns={'field_label': 'field'}))

# papers using term per (year, field)
df_long_2 = (pd.DataFrame.sparse.from_spmatrix(X, columns=terms)
           .assign(year=keyness_docs_df['year'].values,
                   field=keyness_docs_df['field_label'].values)
           .groupby(['year','field'])
           .sum()
           .reset_index()
           .melt(id_vars=['year','field'], var_name='term', value_name='n_kw'))

# Arrow parquet does not support pandas Sparse columns; materialize before export.
df_long_2['n_kw'] = df_long_2['n_kw'].astype('int64')

# rate
df_long_2 = (df_long_2.merge(n_per, on=['year','field'])
                      .assign(rate=lambda d: d['n_kw']/d['n_papers']))
df_long_2.to_csv(UNIGRAM_YEARLY_CSV, index=False)
df_long_2.to_parquet(UNIGRAM_YEARLY_PARQUET, index=False)
print('Saved unigram yearly tables ->', UNIGRAM_YEARLY_CSV, 'and', UNIGRAM_YEARLY_PARQUET)


## Stage 6. Output summary


In [ ]:
support_files = sorted(str(path.name) for path in SUPPORT_DIR.glob('*'))
print('Canonical output:')
print(' -', UNIGRAM_YEARLY_PARQUET.name)
print('\nSupport outputs:')
for name in support_files:
    print(' -', name)
